# eg5 (v2) — Construction XML → Polars DataFrame (data-in)

The DataFrame is built from XML only. The host is asked for the XML once (`xml_out`); `ConstructionIO` never talks to the applet (teacher's ruling 2026-09-08: option I, `getValueString` discarded, no compatibility layer).

In [1]:
import sys, time; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
import polars as pl
from ggblab_extra import ConstructionIO, read_ggb
import ggblab.host.html_host as H; H.DEPLOY = 'https://cdn.geogebra.org/apps/deployggb.js'
from ggblab import GeoGebra

## 1. from a .ggb file (offline)

In [2]:
df1 = ConstructionIO.from_ggb_file('2025_13_01.ggb'); print(df1.shape); df1.head(8)

## 2. from the applet: `xml_in` (setXML) then `xml_out` (getXML) — one round trip each

In [3]:
g = GeoGebra(appName='suite', showToolBar=True, showAlgebraInput=True, showMenuBar=True); g

In [4]:
t0 = time.time(); r = g.set_xml(read_ggb('2025_13_01.ggb'), timeout=60); xml = g.xml(timeout=60)
df2 = ConstructionIO.from_xml(xml); print('XML_LEN', len(xml), 'rows', df2.height, round(time.time()-t0, 2), 's'); df2.head(8)

## 3. the two DataFrames agree (rows aligned by Name; Sequence is the construction order)

In [5]:
a = df1.drop('Sequence').sort('Name'); b = df2.drop('Sequence').sort('Name')
print('EQUAL', a.equals(b), '| types', sorted(set(df2['Type'].to_list()))[:8])
mask = a['Command'].eq_missing(b['Command']).not_(); print('command mismatches', mask.sum())

## 4. pure helpers on the DataFrame (carried from v1)

In [6]:
print(ConstructionIO.commands_for_magic(df2, as_string=False)[:5])
print(ConstructionIO.commands_for_magic(df2, as_string=False, use_name_equals=True)[:3])
print('DONE')